# Section 5 – Package the models and validate the input
**STUDENT notebook: complete the TODO cells**

| | |
|---|---|
| Connects to | Topic 4 – Pydantic, Topic 5 – Model serialization & serving |
| Estimated time | 60 minutes |
| Produces | the `app/` package: `config.py`, `schemas.py`, `model_service.py` |

**Goal:** Put both trained models behind one Python class that loads them once, and protect them with Pydantic models so only valid stats ever reach a model.

### How to work in this notebook
- 🟦 **Worked sample** cells run as they are. Read them, run them, change them.
- 🟨 **Your turn** cells contain `# TODO` and `...`. Replace `...` with your code.
- ✅ **Check** cells print `PASS` / `FAIL`. Do not move on while you see a `FAIL`.
- Stuck? Open the same file in the `solution/` folder, but try first.

## Step 0 – Setup

In [ ]:
import importlib.util
import subprocess
import sys

# 1) Make sure the libraries exist in THIS kernel (installs only what is missing)
REQUIRED = {"requests": "requests", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn", "matplotlib": "matplotlib", "pillow": "PIL",
            "joblib": "joblib", "fastapi": "fastapi", "uvicorn": "uvicorn",
            "httpx": "httpx", "pydantic": "pydantic"}
missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing, "-> restart the kernel if an import below fails")
else:
    print("All dependencies are available")

# 2) Corporate networks with HTTPS inspection: use the Windows certificate store if available
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore active (Windows certificate store)")
except ImportError:
    pass

# 3) Lab folders (this notebook lives in student/ or solution/, so the lab root is one level up)
from pathlib import Path
import json

ROOT = Path.cwd().parent
DATA, MODELS, OUT = ROOT / "data", ROOT / "models", ROOT / "outputs"
for folder in (DATA, MODELS, OUT):
    folder.mkdir(exist_ok=True)


def check(name, ok, detail=""):
    """Self-check helper: prints PASS / FAIL."""
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


print("Lab root:", ROOT)

## The idea
Sections 3 and 4 left two files in `models/`. A notebook is not a service. We now create a small **Python package**
(`app/`) with three single-purpose modules:

```
app/
  __init__.py
  config.py          where the files live (paths)
  schemas.py         the API contract + validation rules (Pydantic)
  model_service.py   loads both models ONCE and answers predict(stats)
```
Why load once? Reading a model from disk on every request is slow. We measure it in Step 6.

## Step 1 – Pydantic in 2 minutes
A Pydantic model is a class where every attribute has a **type** and optional **rules** (`Field`).
Invalid data raises a `ValidationError` that says *which field* failed and *why*.

🟦 **Worked sample** (a toy hero):

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Hero(BaseModel):
    name: str = Field(min_length=1, max_length=20)
    level: int = Field(ge=1, le=100, description="1 to 100")


ok = Hero(name="Ayla", level=12)
print(ok, "->", ok.model_dump())
print("text '7' is converted to the number:", Hero(name="Bo", level="7").level)

try:
    Hero(name="", level=500)
except ValidationError as e:
    for err in e.errors():
        print("  field:", err["loc"][0], "| problem:", err["msg"])

🟨 **Your turn.** Capture the error messages of an invalid hero as a list of `"field: message"` strings.

In [ ]:
messages = []
try:
    Hero(name="Zed", level=0)
except ValidationError as e:
    # TODO: fill `messages` with one string per error, formatted "<field>: <message>"
    # HINT: e.errors() is a list of dicts with the keys "loc" and "msg"
    ...  # <- write your code here

print(messages)
check("exactly one error, about level", len(messages) == 1 and messages[0].startswith("level"))

## Step 2 – Create the package folder

In [ ]:
Path("app").mkdir(exist_ok=True)
print("package folder:", Path("app").resolve())

In [ ]:
%%writefile app/__init__.py
"""Character Card Studio package."""

In [ ]:
%%writefile app/config.py
"""Paths shared by all modules. The lab root is two folders above this file."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"

## Step 3 – The API contract: `schemas.py`
This file is the **contract** from Topic 2, written as code.

| Model | Role |
|---|---|
| `CharacterStats` | INPUT: the 8 numbers. Ranges stop impossible values. |
| `PredictionResponse` | OUTPUT of a prediction |
| `CharacterDetail` | OUTPUT of `GET /v1/characters/{id}` (stats + real value + prediction) |

Valid ranges (game rules): stats `1–255`, height `1–1000` (decimetres), weight `1–10000` (hectograms).
Section 2 saved the real min and max in `feature_info.json`. Compare them with the ranges you choose.

In [ ]:
%%writefile app/schemas.py
"""Pydantic models = the API contract (Topic 2) with validation (Topic 4)."""
from typing import Literal, Optional

from pydantic import BaseModel, Field


class CharacterStats(BaseModel):
    """The 8 numbers every model needs."""
    # TODO: one field per feature, type int, with ge/le ranges: the six stats 1-255, height 1-1000, weight 1-10000
    # HINT: hp: int = Field(ge=1, le=255, description="Hit points")
    ...  # <- write your code here

    model_config = {"json_schema_extra": {"examples": [
        {"hp": 106, "attack": 110, "defense": 90, "special_attack": 154,
         "special_defense": 90, "speed": 130, "height": 20, "weight": 1220}]}}


class PredictionResponse(BaseModel):
    predicted_power: float
    legendary_probability: float = Field(ge=0, le=1)
    # TODO: rarity can only be one of three words
    # HINT: Literal["common", "rare", "legendary"]
    ...  # <- write your code here
    model_version: str


class CharacterDetail(BaseModel):
    id: int
    name: str
    type1: str
    type2: str
    generation: int
    artwork_url: Optional[str] = None
    stats: CharacterStats
    actual_power: int
    is_legendary: bool
    prediction: PredictionResponse

## Step 4 – The model service: `model_service.py`
One class, two responsibilities:
1. In `__init__`: load both models and their metadata (**once**).
2. In `predict`: take a dict of stats, return power + probability + rarity.

Rarity rule (agreed in Section 4): probability `≥ 0.5` legendary, `≥ 0.2` rare, otherwise common.
**Order of the columns matters**: the model was trained with the columns in a specific order, so we build the
`DataFrame` using `self.features`.

In [ ]:
%%writefile app/model_service.py
"""Loads the trained models ONCE and turns stats into predictions."""
import json

import joblib
import pandas as pd

from .config import MODELS_DIR


def rarity_from_probability(p: float) -> str:
    # TODO: 0.5 or more -> "legendary", 0.2 or more -> "rare", otherwise "common"
    ...  # <- write your code here


class ModelService:
    def __init__(self):
        self.regressor = joblib.load(MODELS_DIR / "regressor.joblib")
        self.classifier = joblib.load(MODELS_DIR / "classifier.joblib")
        reg_meta = json.loads((MODELS_DIR / "regressor_meta.json").read_text(encoding="utf-8"))
        clf_meta = json.loads((MODELS_DIR / "classifier_meta.json").read_text(encoding="utf-8"))
        self.features = reg_meta["features"]
        self.version = f"{reg_meta['name']} + {clf_meta['name']}"
        self.power_mae = reg_meta["metrics"]["mae"]

    def predict(self, stats: dict) -> dict:
        """stats: dict with the 8 features. Returns the prediction as a dict."""
        # TODO: a DataFrame with ONE row whose columns follow self.features
        # HINT: pd.DataFrame([[stats[f] for f in self.features]], columns=self.features)
        ...  # <- write your code here
        # TODO: power from the regressor, probability of class 1 from the classifier
        ...  # <- write your code here
        return {
            "predicted_power": round(power, 1),
            "legendary_probability": round(probability, 3),
            "rarity": rarity_from_probability(probability),
            "model_version": self.version,
        }

## Step 5 – Import the package and test it
We import the files we just wrote. (If you edit a file, re-run this cell: it clears the old import first.)

In [ ]:
import sys
import importlib
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[name]                       # forget the old version

from pydantic import ValidationError
from app.schemas import CharacterStats, PredictionResponse, CharacterDetail
from app.model_service import ModelService, rarity_from_probability

service = ModelService()
print("loaded models:", service.version, "| features:", service.features)

In [ ]:
# 1) Rarity rule
check("0.05 -> common", rarity_from_probability(0.05) == "common")
check("0.20 -> rare", rarity_from_probability(0.20) == "rare")
check("0.90 -> legendary", rarity_from_probability(0.90) == "legendary")

# 2) Validation: every real character must pass, impossible values must fail
clean = pd.read_csv(DATA / "pokemon_clean.csv")
failures = 0
for _, row in clean.iterrows():
    try:
        CharacterStats(**{f: int(row[f]) for f in service.features})
    except ValidationError:
        failures += 1
check("all real characters pass the schema", failures == 0, f"{failures} rejected")

for bad in [{"attack": 0}, {"weight": 99999}, {"speed": "fast"}]:
    payload = {"hp": 50, "attack": 50, "defense": 50, "special_attack": 50, "special_defense": 50,
               "speed": 50, "height": 10, "weight": 100, **bad}
    try:
        CharacterStats(**payload)
        check(f"rejects {bad}", False)
    except ValidationError:
        check(f"rejects {bad}", True)

# 3) Predictions
mewtwo = CharacterStats(hp=106, attack=110, defense=90, special_attack=154, special_defense=90, speed=130, height=20, weight=1220)
magikarp = CharacterStats(hp=20, attack=10, defense=55, special_attack=15, special_defense=20, speed=80, height=9, weight=100)
p_mewtwo, p_magikarp = service.predict(mewtwo.model_dump()), service.predict(magikarp.model_dump())
print("mewtwo  :", p_mewtwo)
print("magikarp:", p_magikarp)
PredictionResponse(**p_mewtwo)                       # the output also respects the contract
check("mewtwo is legendary", p_mewtwo["rarity"] == "legendary")
check("magikarp is common", p_magikarp["rarity"] == "common")
check("mewtwo has more power than magikarp", p_mewtwo["predicted_power"] > p_magikarp["predicted_power"])

## Step 6 – Why we load the models once
🟦 **Measure it.** Compare "load on every request" with "load once".

In [ ]:
import time
import joblib

payload = mewtwo.model_dump()

t0 = time.perf_counter()
for _ in range(20):
    ModelService().predict(payload)          # loads from disk every time
slow = (time.perf_counter() - t0) / 20

t0 = time.perf_counter()
for _ in range(20):
    service.predict(payload)                 # model already in memory
fast = (time.perf_counter() - t0) / 20

print(f"load every time: {slow * 1000:.1f} ms per prediction")
print(f"load once      : {fast * 1000:.1f} ms per prediction  ({slow / fast:.0f}x faster)")

## Questions
1. Which Pydantic rule rejected `{"attack": 0}`, and what would the API answer to the client? (Hint: Topic 2, status code)
2. What would happen if `predict` built the DataFrame with the columns in a different order?
3. Why is the contract (`schemas.py`) a separate file from the logic (`model_service.py`)?
4. The API output uses `rarity: Literal[...]`. What does that protect the **UI** from?

### Extra challenge
Add a `power_margin` field to `PredictionResponse` (the MAE of the regressor) so the card can print "power ± error".

**Next:** Section 6 – expose everything with FastAPI.